# Proyecto Integrador Grupal

Redes Neuronales | Universidad de Palermo

Detección, Segmentación y Estimación de Pose utilizando YOLO26

---

**Grupo 03: Integrantes**
- Aaron Fraga
- Juan Cruz Rey
- Juan Ignacio Pintarelli
- Santiago Minnaar Vizzotto
- Paula Goldenhorn

### Resultados

En este Notebook realizaremos una combinación de inferencias sobre un video propio para dar como resultado un video con Detección, Segmentación y Estimación de Pose. Utilizaremos el modelo que entrenamos sobre nuestros datos y dos modelos de YOLO (sin fine-tuning) para observar el resultado final del proyecto.

## Imports

In [1]:
# -- Plataforma para usar modelo preentrenado YOLO --
!pip install ultralytics --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.7 MB/s eta 0:00:00


In [2]:
# -- Importación de librerías --
import numpy as np
import torch
from collections import defaultdict

from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator

import cv2
from google.colab.patches import cv2_imshow

from PIL import Image
from pathlib import Path
from IPython.display import Video, Image, display
import subprocess

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# --- Detección de device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"VRAM libre:     {torch.cuda.memory_reserved(0) / 1e9:.2f} GB reservada")
    DEVICE_ARG = 0  # índice de GPU para model.train()
else:
    print("⚠️  Sin GPU — el entrenamiento será muy lento, reducir epochs a ≤ 5.")
    DEVICE_ARG = "cpu"

Device: cuda
GPU: Tesla T4
VRAM total:     15.6 GB
VRAM libre:     0.00 GB reservada


## Carga de modelos YOLO

Cargamos los modelos YOLO 26:

* `model_detection`: es el modelo YOLO 2026 Nano que entrenamos para detectar barra, disco y mancuerna
* `model_segmentation`: es el modelo YOLO 2026 Nano que segmenta objetos conocidos dentro de sus 80 clases
* `model_pose`: es el modelo YOLO 2026 Nano que detecta la pose de las persona

In [4]:
model_detection = YOLO("bestV27.pt")
model_segmentation = YOLO("yolo26n-seg.pt")
model_pose = YOLO("yolo26n-pose.pt")

## Metodos para procesar videos

In [5]:
CLASS_COLORS = {
    "barra": (255, 255, 0),       # celeste
    "disco": (255, 0, 140),       # violeta
    "mancuerna": (174, 0, 255),   # magenta
}

#### Bounding Boxes + Tracking

In [6]:
LARGO_ESTELA = 90
track_history = defaultdict(lambda: [])

def boxes_drawer(results, frame):

    annotator = Annotator(
        frame,
        line_width=2,
        font_size=12
    )

    if results[0].boxes.id is None:
        return frame

    for detected_box in results[0].boxes:

        # Clase detectada
        detected_class_id = int(detected_box.cls[0])
        detected_class_name = model_detection.names[detected_class_id]

        # Confianza
        conf = float(detected_box.conf[0])

        x1, y1, x2, y2 = map(int, detected_box.xyxy[0])

        # Etiqueta
        label = f"{detected_class_name} {conf:.2f}"

        # Color por clase
        color = CLASS_COLORS.get(
            detected_class_name,
            (255, 255, 255)
        )

        # --------------------------------------------------
        # TRACKING DEL DISCO
        # --------------------------------------------------

        color_track = (0, 0, 255) # rojo para la estela

        if detected_class_name == "disco":

            track_id = int(detected_box.id[0])

            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)

            centroid = (center_x, center_y)

            history = track_history[track_id]
            history.append(centroid)

            if len(history) > LARGO_ESTELA:
                history.pop(0)

            cv2.circle(frame, centroid, 5, color_track, -1)
            cv2.circle(frame, centroid, 2, (255, 255, 255), -1)

            if len(history) > 1:

                points = (
                    np.array(history)
                    .reshape((-1, 1, 2))
                    .astype(np.int32)
                )

                cv2.polylines(
                    frame,
                    [points],
                    isClosed=False,
                    color=color_track,
                    thickness=2
                )

        # --------------------------------------------------
        # BOUNDING BOX + ETIQUETA YOLO
        # --------------------------------------------------

        annotator.box_label(
            [x1, y1, x2, y2],
            label,
            color=color
        )

    return annotator.result()

#### Segmentation

In [7]:
def segmentation_drawer(results, frame):

    if results[0].masks is None:
        return frame

    overlay = frame.copy()

    labels_to_draw = []

    for detected_mask, detected_box in zip(
        results[0].masks.xy,
        results[0].boxes
    ):

        detected_class_id = int(detected_box.cls[0])

        detected_class_name = model_segmentation.names[
            detected_class_id
        ]

        # NO segmentar personas
        if detected_class_name == "person":
            continue

        polygon_points = np.array(
            detected_mask,
            dtype=np.int32
        )

        # Color de la máscara
        color = (0, 255, 0) # verde

        cv2.fillPoly(
            overlay,
            [polygon_points],
            color
        )

        x1, y1, x2, y2 = detected_box.xyxy[0]

        labels_to_draw.append(
            (
                detected_class_name,
                (int((x1 + x2) / 2), int((y1 + y2) / 2)) # ubicar etiquetas al centro
            )
        )

    # Aplicar transparencia a las máscaras
    blended_frame = cv2.addWeighted(
        overlay,
        0.4,
        frame,
        0.6,
        0
    )

    # Dibujar etiquetas DESPUÉS del blend
    for label, position in labels_to_draw:

        draw_text_with_background(
            blended_frame,
            label,
            position,
            text_color=(255, 255, 255),
            background_color=(0, 120, 0),
            font_scale=0.6,
            thickness=2
        )

    return blended_frame

#### Pose Estimation

In [8]:
# -- Pose estimation --
def pose_drawer(results, frame):

    keypoint_color = (0, 255, 0)
    skeleton_color = (255, 0, 0)

    if results[0].keypoints is None:
        return frame

    skeleton_connections = [
        (0, 1), (0, 2), (1, 3), (2, 4),
        (5, 6), (5, 7), (7, 9), (6, 8),
        (8, 10), (5, 11), (6, 12),
        (11, 12), (11, 13), (13, 15),
        (12, 14), (14, 16)
    ]

    # Dibujar poses
    for detected_person_keypoints in results[0].keypoints.xy:

        # Keypoints
        for keypoint_x, keypoint_y in detected_person_keypoints:

            cv2.circle(
                frame,
                (int(keypoint_x), int(keypoint_y)),
                4,
                keypoint_color,
                -1
            )

        # Esqueleto
        for start_index, end_index in skeleton_connections:

            start_x, start_y = detected_person_keypoints[start_index]
            end_x, end_y = detected_person_keypoints[end_index]

            cv2.line(
                frame,
                (int(start_x), int(start_y)),
                (int(end_x), int(end_y)),
                skeleton_color,
                2
            )

    # Dibujar IDs del tracker
    color_id = (0, 0, 255) # rojo

    if results[0].boxes is not None:

        for box in results[0].boxes:

            if box.id is None:
                continue

            track_id = int(box.id[0])

            x1, y1, x2, y2 = box.xyxy[0]

            cv2.putText(
                frame,
                f"ID {track_id}",
                (int(x1), int(y1) - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color_id,
                2
            )

    return frame

#### Contador de objetos

In [9]:
# -- Fondo de los textos --
def draw_text_with_background(
    frame,
    text,
    position,
    text_color,
    background_color=(40, 40, 40),
    font_scale=0.6,
    thickness=2,
    padding=4
):
    font = cv2.FONT_HERSHEY_SIMPLEX

    (w, h), baseline = cv2.getTextSize(
        text,
        font,
        font_scale,
        thickness
    )

    x, y = position

    cv2.rectangle(
        frame,
        (x - padding, y - h - padding),
        (x + w + padding, y + baseline + padding),
        background_color,
        -1
    )

    cv2.putText(
        frame,
        text,
        (x, y),
        font,
        font_scale,
        text_color,
        thickness
    )

In [10]:
# -- Estadísticas de detección --
def statistics_drawer(detection_results, frame):

    detected_barra = 0
    detected_disco = 0
    detected_mancuerna = 0

    for detected_box in detection_results[0].boxes:

        detected_class_id = int(detected_box.cls[0])

        detected_class_name = model_detection.names[detected_class_id]

        if detected_class_name == "barra":
            detected_barra += 1

        elif detected_class_name == "disco":
            detected_disco += 1

        elif detected_class_name == "mancuerna":
            detected_mancuerna += 1

    overlay = frame.copy()

    panel_start_point = (10, 10)
    panel_end_point = (260, 110)

    color_rectangle = (30, 30, 30)

    cv2.rectangle(
        overlay,
        panel_start_point,
        panel_end_point,
        color_rectangle,
        -1
    )

    blended_frame = cv2.addWeighted(
        overlay,
        0.6,
        frame,
        0.4,
        0
    )

    # Textos
    draw_text_with_background(
        blended_frame,
        f"Barras: {detected_barra}",
        (20, 35),
        CLASS_COLORS["barra"]
    )

    draw_text_with_background(
        blended_frame,
        f"Discos: {detected_disco}",
        (20, 65),
        CLASS_COLORS["disco"]
    )

    draw_text_with_background(
        blended_frame,
        f"Mancuernas: {detected_mancuerna}",
        (20, 95),
        CLASS_COLORS["mancuerna"]
    )

    return blended_frame

#### Procesamiento

Se eligió un umbral de confianza de **0.35** para la segmentación.


In [11]:
# -- Procesamiento del video --
def process_video(video_path, output_path):

    capture = cv2.VideoCapture(video_path)

    fps = capture.get(cv2.CAP_PROP_FPS)

    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))

    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

    codec = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        output_path,
        codec,
        fps,
        (width, height)
    )

    while capture.isOpened():

        readed, frame = capture.read()

        if not readed:
            break

        # Inferencia
        detection_results = model_detection.track(
            frame,
            persist=True,
            verbose=False

        )

        segmentation_results = model_segmentation(
            frame,
            verbose=False,
            conf=0.35
        )

        pose_results = model_pose.track(
            frame,
            persist=True,
            verbose=False
        )

        # Dibujar detecciones
        frame_with_boxes = boxes_drawer(
            detection_results,
            frame.copy()
        )

        # Dibujar segmentación
        frame_with_segmentation = segmentation_drawer(
            segmentation_results,
            frame_with_boxes
        )

        # Dibujar poses
        frame_with_pose = pose_drawer(
            pose_results,
            frame_with_segmentation
        )

        # Dibujar estadísticas
        processed_frame = statistics_drawer(
            detection_results,
            frame_with_pose
        )

        # Guardar frame
        writer.write(processed_frame)

    capture.release()

    writer.release()

## Inferencia: Video propio

### Video 1

In [ ]:
VIDEO_PATH = "/content/video_original_1.mp4"
VIDEO_OUTPUT_PATH = "/content/video_original_1_result.mp4"
FINAL_OUTPUT_PATH = "/content/video_original_1_final.mp4"

# Procesar video
process_video(
    video_path=VIDEO_PATH,
    output_path=VIDEO_OUTPUT_PATH
)

# Convertir a formato compatible navegador
subprocess.run([
    "ffmpeg",
    "-y",
    "-i", VIDEO_OUTPUT_PATH,
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    FINAL_OUTPUT_PATH
], check=True)

# Mostrar video embebido
Video(FINAL_OUTPUT_PATH, embed=True)

### Video 2

In [ ]:
VIDEO_PATH = "/content/video_original_2.mp4"
VIDEO_OUTPUT_PATH = "/content/video_original_2_result.mp4"
FINAL_OUTPUT_PATH = "/content/video_original_2_final.mp4"

# Procesar video
process_video(
    video_path=VIDEO_PATH,
    output_path=VIDEO_OUTPUT_PATH
)

# Convertir a formato compatible navegador
subprocess.run([
    "ffmpeg",
    "-y",
    "-i", VIDEO_OUTPUT_PATH,
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    FINAL_OUTPUT_PATH
], check=True)

# Mostrar video embebido
Video(FINAL_OUTPUT_PATH, embed=True)

### Video 3: Demo en vivo

Tarda ~ 55 seg

In [ ]:
VIDEO_PATH = "/content/video_original_demo.mp4"
VIDEO_OUTPUT_PATH = "/content/video_original_demo_result.mp4"
FINAL_OUTPUT_PATH = "/content/video_original_demo_final.mp4"

# Procesar video
process_video(
    video_path=VIDEO_PATH,
    output_path=VIDEO_OUTPUT_PATH
)

# Convertir a formato compatible navegador
subprocess.run([
    "ffmpeg",
    "-y",
    "-i", VIDEO_OUTPUT_PATH,
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    FINAL_OUTPUT_PATH
], check=True)

# Mostrar video embebido
Video(FINAL_OUTPUT_PATH, embed=True)

## Conclusión

En los videos se puede apreciar:
- Segmentación de clases YOLO
- Pose estimation + ID tracker
- Bounding boxes sobre clases preentrenadas
- Reporte de clases
- Tracking de clase "disco"